# 文本摘要示例

## Step1 导入包

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

## Step2 加载数据集

In [ ]:
ds = Dataset.load_from_disk("./nlpcc_2017/")
ds

In [ ]:
ds = ds.train_test_split(100, seed = 42)
ds

In [ ]:
ds["train"][0]

## Step3 数据处理

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Langboat/mengzi-t5-base")
tokenizer

In [ ]:
?tokenizer.__call__

In [ ]:
def process_func(examples):
    contents = ["摘要生成：\n" + example_content for example_content in examples["content"]]
    inputs = tokenizer(contents, max_length=384, truncation=True)
    labels = tokenizer(text_target=examples["title"], max_length=64, truncation=True)
    inputs["labels"]=labels["input_ids"]
    return inputs

In [ ]:
tokenized_ds = ds.map(process_func, batched=True)
print(tokenized_ds["train"][0])
print(tokenizer.decode(tokenized_ds["train"][0]["input_ids"]))
print(tokenizer.decode(tokenized_ds["train"][0]["labels"]))

## Step4 创建模型

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("Langboat/mengzi-t5-base")

## Step5 创建评估函数

In [ ]:
import numpy as np
from rouge_chinese import Rouge

rouge = Rouge()

def compute_metrics(evalPred):
    predictions, labels = evalPred
    # predictions 是模型输出的 token id，需要 decode 成文本
    # predictions 是一个batch，所以需要使用 batch_decode，并且 skip_special_tokens=True 来去掉特殊 token
    decode_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # 不能直接 decode labels，因为 labels 中的 -100 是用来忽略的 token id，不能被 decode 成文本
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decode_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decode_preds = [" ".join(pred) for pred in decode_preds]
    decode_labels = [" ".join(label) for label in decode_labels]
    scores = rouge.get_scores(decode_preds,decode_labels,avg=True)
    return {
        "rouge-1": scores["rouge-1"]["f"],
        "rouge-2": scores["rouge-2"]["f"],
        "rouge-l": scores["rouge-l"]["f"],
    }



## Step6 创建训练参数

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir="./summary",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    logging_steps=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="rouge-1",
    predict_with_generate=True,  # Seq2SeqTrainingArguments 特有的参数
)

## Step7 创建训练器

In [ ]:
trainer = Seq2SeqTrainer(
    args=args,
    model=model,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["test"],
    compute_metrics=compute_metrics,
    data_collator=DataCollatorForSeq2Seq(tokenizer)
)

## Step8 训练模型

In [ ]:
trainer.train()

## Step9 模型推理

In [ ]:
input_text = ds["test"][0]["content"]
inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
outputs = model.generate(**inputs, max_length=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))